# Enrollment Prediction Model Training - Kaggle

Train BioBERT-based enrollment prediction model using Kaggle's free GPU.

**Estimated time: 30-45 minutes on P100 GPU**

## Setup Instructions:
1. **Enable GPU**: Settings → Accelerator → GPU P100
2. **Enable Internet**: Settings → Internet → ON (for ChromaDB access)
3. **Upload files**: Click 'Add Data' → Upload `enrollment_predictor.py` and `training.py`
4. **Set ChromaDB credentials** in cell below
5. **Run all cells**
6. **Download trained model** at the end

## 1. Install Dependencies

In [ ]:
!pip install -q transformers==4.38.0 chromadb scikit-learn tqdm
print("✓ Dependencies installed")

## 2. Verify GPU

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print("\n✅ GPU is ready!")
else:
    print("\n⚠️ GPU not detected! Enable GPU in Settings → Accelerator")

## 3. Set ChromaDB Credentials

**⚠️ IMPORTANT: Replace with your actual credentials**

In [ ]:
import os

# Set your ChromaDB credentials here
os.environ['CHROMA_API_KEY'] = 'your_api_key_here'
os.environ['CHROMA_TENANT'] = 'your_tenant_here'
os.environ['CHROMA_DATABASE'] = 'ClinicalAgents'
os.environ['CHROMA_COLLECTION'] = 'clinical_trials'

print("✓ Credentials set")

## 4. Load Uploaded Files

Upload `enrollment_predictor.py` and `training.py` using 'Add Data' → 'Upload' in the right sidebar

In [ ]:
import sys
import shutil
from pathlib import Path

# Create ml_models directory
!mkdir -p ml_models

# Copy uploaded files from Kaggle input directory
input_dir = Path('/kaggle/input')

# Find the uploaded files (they'll be in a subdirectory)
for item in input_dir.rglob('*.py'):
    if item.name in ['enrollment_predictor.py', 'training.py']:
        shutil.copy(item, f'ml_models/{item.name}')
        print(f"✓ Copied {item.name}")

# Create __init__.py
with open('ml_models/__init__.py', 'w') as f:
    f.write('from .enrollment_predictor import *\n')
    f.write('from .training import *\n')

# Add to path
sys.path.insert(0, '/kaggle/working')

print("\n✓ Files organized and ready!")

## 5. Train Model

This will take ~30-45 minutes on Kaggle P100 GPU

In [ ]:
from ml_models.training import train_enrollment_model

print("🚀 Starting training...\n")

# Train the model
history = train_enrollment_model(
    collection_name='clinical_trials',
    epochs=10,
    batch_size=32,  # Larger batch size for P100 GPU
    learning_rate=2e-5,
    max_samples=None,  # Use all data
    save_dir='saved_models',
    freeze_bert=False
)

print("\n" + "="*60)
print("✨ TRAINING COMPLETE!")
print("="*60)
print(f"\nBest F1 Score: {max(history['val_f1']):.4f}")
print(f"Best Accuracy: {max(history['val_accuracy']):.4f}")

## 6. Create Download Package

In [ ]:
# Zip the model files
!zip -r enrollment_model.zip saved_models/

print("✓ Model packaged!")
print("\n📥 Download 'enrollment_model.zip' from the Output section on the right")
print("   Extract to: agents_server/ml_models/saved_models/")

## 7. Test the Model (Optional)

In [ ]:
from ml_models.enrollment_predictor import EnrollmentFusionModel
from transformers import AutoTokenizer
import torch
import numpy as np

# Load model
checkpoint = torch.load('saved_models/enrollment_model.pt', map_location='cuda', weights_only=False)
model = EnrollmentFusionModel()
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✓ Model loaded successfully!")
print(f"\nModel Info:")
print(f"  Classes: success, delayed, fail")
print(f"  Feature mean: {checkpoint['feature_mean']}")
print(f"  Feature std: {checkpoint['feature_std']}")
print(f"  Class weights: {checkpoint.get('class_weights', 'N/A')}")

## 8. Training Metrics Visualization (Optional)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['val_accuracy'], label='Val Accuracy', color='green')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

# F1 Score
axes[2].plot(history['val_f1'], label='Val F1', color='orange')
axes[2].set_title('Validation F1 Score')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('F1 Score')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f"\n📊 Training Summary:")
print(f"  Best Epoch: {np.argmax(history['val_f1']) + 1}")
print(f"  Best F1: {max(history['val_f1']):.4f}")
print(f"  Best Accuracy: {max(history['val_accuracy']):.4f}")
print(f"  Final Train Loss: {history['train_loss'][-1]:.4f}")
print(f"  Final Val Loss: {history['val_loss'][-1]:.4f}")

---

## ✅ Next Steps

1. **Download** `enrollment_model.zip` from the Output section
2. **Extract** to `agents_server/ml_models/saved_models/`
3. **Test** locally: `python test_model.py`
4. **Integrate** into your enrollment agent

---

**Kaggle Advantages:**
- ✅ Free P100 GPU (16GB VRAM)
- ✅ 30 hours/week GPU quota
- ✅ Faster than Colab free tier
- ✅ Better for longer training runs